<a href="https://colab.research.google.com/github/Sayrinix/Entrega_Final_CDD_Luciano_Sayrina/blob/main/notebooks/S2_Luciano_Sayrina_CDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | _(Sayrina Luciano Jofré)_ |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

**Pregunta analítica**

¿Cómo se distribuyen los montos netos en pesos chilenos de las órdenes de compra entre los Gobiernos Regionales de Coquimbo, Metropolitana y Valparaíso durante el primer semestre de 2026, según el mes y el mecanismo de contratación?

**Unidad de análisis:** cada orden de compra identificada mediante `codigoOC`.

**Fuente:** 12 archivos CSV de ChileCompra, correspondientes a los cuatro mecanismos de contratación (CompraAgil, ConvenioMarco, Licitacion, TratoDirecto) seleccionados de las Regiones Metropolitana, Coquimbo y Valparaíso.

### Datos originales y reproducibilidad

Este laboratorio utiliza datos abiertos de ChileCompra correspondientes a órdenes de compra de los Gobiernos Regionales de Coquimbo, Metropolitana y Valparaíso, durante el primer semestre de 2026.

Los archivos originales utilizados en el análisis se encuentran reunidos en `ChileCompra.zip`, disponible en el repositorio de GitHub del proyecto.

Para reproducir el análisis, descargue el archivo ZIP desde el repositorio, cárguelo en Google Colab y ejecute las celdas en el orden establecido.

**Fuente original:** [Datos Abiertos de ChileCompra](https://datos-abiertos.chilecompra.cl/descargas).

**Archivo utilizado:** [ChileCompra.zip](https://github.com/Sayrinix/chilecompra-calidad-datos/blob/main/datos/ChileCompra.zip).

In [ ]:
import zipfile
import os

archivo_zip = "/content/ChileCompra.zip"
destino = "/content/ChileCompra"

with zipfile.ZipFile(archivo_zip, "r") as zip_ref: zip_ref.extractall(destino)

print("Archivos descomprimidos correctamente")

import glob

archivos_csv = glob.glob(
    "/content/ChileCompra/**/*.csv",
    recursive=True
)

print("Cantidad de archivos CSV:", len(archivos_csv))

for archivo in archivos_csv:
    print(archivo)

Archivos descomprimidos correctamente
Cantidad de archivos CSV: 12
/content/ChileCompra/ChileCompra/datos_originales/Coquimbo/26940ConvenioMarco.csv
/content/ChileCompra/ChileCompra/datos_originales/Coquimbo/26940OCLicitacion.csv
/content/ChileCompra/ChileCompra/datos_originales/Coquimbo/26940TratoDirecto.csv
/content/ChileCompra/ChileCompra/datos_originales/Coquimbo/26940CompraAgil.csv
/content/ChileCompra/ChileCompra/datos_originales/Metropolitana/27222ConvenioMarco.csv
/content/ChileCompra/ChileCompra/datos_originales/Metropolitana/27222OCLicitacion.csv
/content/ChileCompra/ChileCompra/datos_originales/Metropolitana/27222TratoDirecto.csv
/content/ChileCompra/ChileCompra/datos_originales/Metropolitana/27222CompraAgil.csv
/content/ChileCompra/ChileCompra/datos_originales/Valparaiso/27172CompraAgil.csv
/content/ChileCompra/ChileCompra/datos_originales/Valparaiso/27172ConvenioMarco.csv
/content/ChileCompra/ChileCompra/datos_originales/Valparaiso/27172OCLicitacion.csv
/content/ChileCompr

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

SEMILLA = 20262
np.random.seed(SEMILLA)

tablas = []
control_archivos = []

for ruta in sorted(archivos_csv):
    ruta = Path(ruta)

    # Identificar procedencia
    region = ruta.parent.name
    mecanismo = next(
    nombre for nombre in [
        "CompraAgil",
        "ConvenioMarco",
        "Licitacion",
        "TratoDirecto"
    ]
    if ruta.stem.endswith(nombre)
)

    # Leer el archivo original
    temporal = pd.read_csv(
        ruta,
        sep=";",
        encoding="cp1252"
    )

    # Guardar información de procedencia
    temporal["RegionOrigen"] = region
    temporal["MecanismoCompra"] = mecanismo

    control_archivos.append({
        "region": region,
        "mecanismo": mecanismo,
        "filas": len(temporal),
        "columnas_originales": len(temporal.columns) - 2
    })

    tablas.append(temporal)

# Consolidar los archivos
df = pd.concat(tablas, ignore_index=True)

# Mostrar resultados
control = pd.DataFrame(control_archivos)

display(control)
print("Dimensiones consolidadas:", df.shape)
print("Órdenes únicas:", df["codigoOC"].nunique())


,region,mecanismo,filas,columnas_originales
0,Coquimbo,CompraAgil,248,63
1,Coquimbo,ConvenioMarco,120,59
2,Coquimbo,Licitacion,39,46
3,Coquimbo,TratoDirecto,7,44
4,Metropolitana,CompraAgil,249,63
5,Metropolitana,ConvenioMarco,36,59
6,Metropolitana,Licitacion,44,46
7,Metropolitana,TratoDirecto,17,44
8,Valparaiso,CompraAgil,65,63
9,Valparaiso,ConvenioMarco,94,59


Dimensiones consolidadas: (940, 81)
Órdenes únicas: 336


In [ ]:
print("Filas de los archivos:", control["filas"].sum())
print("Filas consolidadas:", len(df))

print(
    "Mismas columnas originales:",
    control["columnas_originales"].unique()
)

print("Regiones:", df["RegionOrigen"].unique())
print("Mecanismos:", df["MecanismoCompra"].unique())

Filas de los archivos: 940
Filas consolidadas: 940
Mismas columnas originales: [63 59 46 44]
Regiones: ['Coquimbo' 'Metropolitana' 'Valparaiso']
Mecanismos: ['CompraAgil' 'ConvenioMarco' 'Licitacion' 'TratoDirecto']


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [ ]:
faltantes = df.isna().sum().to_frame("faltantes")

faltantes["pct"] = (
    faltantes["faltantes"] / len(df) * 100
).round(2)

faltantes.sort_values("faltantes", ascending=False)

,faltantes,pct
NroConvenioMarco,940,100.00
Modelo,940,100.00
Marca,940,100.00
Producto,940,100.00
MontoAprobGC_SubTotalIVA,939,99.89
...,...,...
Institucion,0,0.00
entCode,0,0.00
UnidadCompraRUT,0,0.00
MecanismoCompra,0,0.00


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [ ]:

pd.crosstab(
    df["MecanismoCompra"],
    df["NroConvenioMarco"].isna(),
    normalize="index"
).mul(100).round(2)


NroConvenioMarco,True
MecanismoCompra,
CompraAgil,100.0
ConvenioMarco,100.0
Licitacion,100.0
TratoDirecto,100.0


In [ ]:
columnas_clave = [
    "codigoOC",
    "FechaEnvioOC",
    "Institucion",
    "EstadoOC",
    "MonedaOC",
    "MontoTotalOC",
    "MecanismoCompra",
    "RegionOrigen"
]

faltantes.loc[columnas_clave]

,faltantes,pct
codigoOC,0,0.0
FechaEnvioOC,0,0.0
Institucion,0,0.0
EstadoOC,0,0.0
MonedaOC,0,0.0
MontoTotalOC,0,0.0
MecanismoCompra,0,0.0
RegionOrigen,0,0.0


### 1.2 Exactitud — rangos y valores imposibles

In [ ]:
df[
    ["MontoTotalOC", "MonedaOC", "FechaEnvioOC"]
].dtypes

,0
MontoTotalOC,float64
MonedaOC,object
FechaEnvioOC,object


In [ ]:
df["MonedaOC"].value_counts(dropna=False)

,count
MonedaOC,
CLP,909
CLF,19
USD,12


In [ ]:
print("Monto mínimo:", df["MontoTotalOC"].min())
print("Monto máximo:", df["MontoTotalOC"].max())

print(
    "Montos negativos:",
    (df["MontoTotalOC"] < 0).sum()
)

print(
    "Montos iguales a cero:",
    (df["MontoTotalOC"] == 0).sum()
)

Monto mínimo: 13.447
Monto máximo: 2399887695.9883
Montos negativos: 0
Montos iguales a cero: 0


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [ ]:
# Tu código aquí. Sugerencia:
# for col in df.select_dtypes(include='number').columns:
#     print(col, sorted(df[col].dropna().unique())[:5], '...', sorted(df[col].dropna().unique())[-5:])

centinelas = [-1, 999, 9999, 99999]

for valor in centinelas:
    cantidad = (df["MontoTotalOC"] == valor).sum()
    print(f"Valor {valor}: {cantidad} apariciones")

Valor -1: 0 apariciones
Valor 999: 0 apariciones
Valor 9999: 0 apariciones
Valor 99999: 0 apariciones


In [ ]:
print("Tipo:", df["MontoNetoOC_CLP"].dtype)

print(
    "Faltantes:",
    df["MontoNetoOC_CLP"].isna().sum()
)

print(
    "Porcentaje de faltantes:",
    round(df["MontoNetoOC_CLP"].isna().mean() * 100, 2)
)

Tipo: float64
Faltantes: 0
Porcentaje de faltantes: 0.0


### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [ ]:
# Regla 1: Una orden de compra debe tener un identificador de institución compradora coherente.
instituciones_por_orden = df.groupby("codigoOC")["Institucion"].nunique()

print(
    "Órdenes asociadas a más de una institución:",
    (instituciones_por_orden > 1).sum()
)

# Regla 2: Una misma orden de compra no debería presentar diferentes organismos compradores o monedas sin una explicación.
monedas_por_orden = df.groupby("codigoOC")["MonedaOC"].nunique()

print(
    "Órdenes con más de una moneda:",
    (monedas_por_orden > 1).sum()
)

Órdenes asociadas a más de una institución: 0
Órdenes con más de una moneda: 0


### 1.4 Unicidad

In [ ]:
print('Duplicados exactos:', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
# print('ID repetidos:', df['TU_ID'].duplicated().sum())

repeticiones = df["codigoOC"].value_counts()

print(
    "Órdenes con múltiples filas:",
    (repeticiones > 1).sum()
)

print(
    "Máximo de filas por orden:",
    repeticiones.max()
)

Duplicados exactos: 46
Órdenes con múltiples filas: 96
Máximo de filas por orden: 103


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [ ]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Tu código aquí
clave = (
    normalizar_texto(df["codigoOC"])
    + "|"
    + normalizar_texto(df["NombreItem"])
    + "|"
    + df["CantidadItem"].astype(str)
    + "|"
    + df["MontoTotalItem"].astype(str)
)

print(
    "Coincidencias según clave normalizada:",
    clave.duplicated().sum()
)

Coincidencias según clave normalizada: 289


In [ ]:
duplicados_exactos = df.duplicated()

coincidencias_clave = clave.duplicated()

posibles_casi_duplicados = (
    coincidencias_clave & ~duplicados_exactos
)

print(
    "Coincidencias de clave que no son duplicados exactos:",
    posibles_casi_duplicados.sum()
)

Coincidencias de clave que no son duplicados exactos: 243


### 1.5 Oportunidad

In [ ]:
print(df["FechaEnvioOC"].head(15).to_string(index=False))

2026-02-11
2026-02-13
2026-02-13
2026-02-13
2026-02-23
2026-03-13
2026-03-27
2026-04-02
2026-04-21
2026-04-22
2026-04-22
2026-04-22
2026-04-22
2026-04-22
2026-04-22


In [ ]:
# Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?
# ¿El conjunto está vigente para la pregunta que quieres responder?
# Conversión temporal, sin modificar df
fechas = pd.to_datetime(
    df["FechaEnvioOC"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

print("Fecha mínima:", fechas.min())
print("Fecha máxima:", fechas.max())
print("Fechas no interpretables:", fechas.isna().sum())

fuera_periodo = (
    fechas.notna() &
    ~fechas.between("2026-01-01", "2026-06-30 23:59:59")
)

print("Registros fuera del semestre:", fuera_periodo.sum())

Fecha mínima: 2026-01-05 00:00:00
Fecha máxima: 2026-06-30 00:00:00
Fechas no interpretables: 0
Registros fuera del semestre: 0


In [ ]:
columnas_orden = [
    "FechaEnvioOC",
    "Institucion",
    "RegionOrigen",
    "MecanismoCompra",
    "MonedaOC",
    "MontoTotalOC",
    "MontoNetoOC_CLP",
    "EstadoOC"
]

validacion = (
    df.groupby("codigoOC")[columnas_orden]
      .nunique(dropna=False)
)

inconsistencias = (validacion > 1).sum()

print(inconsistencias)

FechaEnvioOC       0
Institucion        0
RegionOrigen       0
MecanismoCompra    0
MonedaOC           0
MontoTotalOC       0
MontoNetoOC_CLP    0
EstadoOC           0
dtype: int64


---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [ ]:
# Tu código aquí
centinelas_texto = ["S/I", "NULL", "SIN DATO"]

for columna in df.select_dtypes(include="object").columns:
    valores = df[columna].astype("string").str.strip().str.upper()
    cantidad = valores.isin(centinelas_texto).sum()

    if cantidad > 0:
        print(columna, ":", cantidad)
    else: print(columna, ":", "0 centinelas")

codigoOC : 0 centinelas
FechaEnvioOC : 0 centinelas
NombreOC : 0 centinelas
DescripcionOC : 0 centinelas
EstadoOC : 0 centinelas
ProcedenciaOC : 0 centinelas
MonedaOC : 0 centinelas
MetodoPago : 0 centinelas
TipoDespacho : 0 centinelas
Financiamiento : 0 centinelas
UnidadCompra : 0 centinelas
UnidadCompraRUT : 0 centinelas
RegionUnidadCompra : 0 centinelas
Institucion : 0 centinelas
Sector : 0 centinelas
Proveedor : 0 centinelas
ProveedorRUT : 0 centinelas
ActividadProveedor : 0 centinelas
TamanoProveedor : 0 centinelas
RegionProveedor : 0 centinelas
RubroN1 : 0 centinelas
RubroN2 : 0 centinelas
RubroN3 : 0 centinelas
ONUProducto : 0 centinelas
NombreItem : 0 centinelas
DescripcionItem : 0 centinelas
UnidadMedida : 0 centinelas
MonedaItem : 0 centinelas
CodigoCotizacion : 0 centinelas
NombreCotizacion : 0 centinelas
DescripcionCotizacion : 0 centinelas
FechaPublicacionCotizacion : 0 centinelas
FechaCierreCotizacion : 0 centinelas
DescripcionProducto : 0 centinelas
ProductoCotizado : 0 

### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [ ]:
# Tu código aquí
print(df["EstadoOC"].value_counts(dropna=False))

estados_normalizados = normalizar_texto(df["EstadoOC"])

print(estados_normalizados.value_counts(dropna=False))

print("Categorías originales:", df["EstadoOC"].nunique())
print("Categorías normalizadas:", estados_normalizados.nunique())

datos_trabajo = df.copy()

datos_trabajo["EstadoOC"] = normalizar_texto(
    datos_trabajo["EstadoOC"]
)

print(datos_trabajo["EstadoOC"].value_counts())

EstadoOC
Recepcion Conforme    715
Aceptada              224
En Proceso              1
Name: count, dtype: int64
EstadoOC
RECEPCION CONFORME    715
ACEPTADA              224
EN PROCESO              1
Name: count, dtype: int64
Categorías originales: 3
Categorías normalizadas: 3
EstadoOC
RECEPCION CONFORME    715
ACEPTADA              224
EN PROCESO              1
Name: count, dtype: int64


### ✏️ Actividad 2.3 — Duplicados

In [ ]:
# Tu código aquí

datos_trabajo = datos_trabajo.copy()
filas_antes = len(datos_trabajo)
ordenes = datos_trabajo.drop_duplicates(
    subset=["codigoOC"],
    keep="first"
).copy()
print("Filas originales:", filas_antes)
print("Órdenes preparadas:", len(ordenes))
print("Códigos duplicados:", ordenes["codigoOC"].duplicated().sum())

Filas originales: 940
Órdenes preparadas: 336
Códigos duplicados: 0


### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [ ]:
# Tu código aquí
columnas = [
    "codigoOC",
    "FechaEnvioOC",
    "MontoTotalOC",
    "MontoNetoOC_CLP",
    "MonedaOC"
]

print(ordenes[columnas].dtypes)

#Conversion de Fecha
ordenes["FechaEnvioOC"] = pd.to_datetime(
    ordenes["FechaEnvioOC"],
    format="%Y-%m-%d",
    errors="coerce"
)

print(ordenes["FechaEnvioOC"].dtype)

print(
    "Fechas inválidas:",
    ordenes["FechaEnvioOC"].isna().sum()
)

codigoOC            object
FechaEnvioOC        object
MontoTotalOC       float64
MontoNetoOC_CLP    float64
MonedaOC            object
dtype: object
datetime64[ns]
Fechas inválidas: 0


### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [ ]:
# Tu código aquí

columna = "MontoNetoOC_CLP"

q1 = ordenes[columna].quantile(0.25)
q3 = ordenes[columna].quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

atipicos = ordenes[
    (ordenes[columna] < limite_inferior) |
    (ordenes[columna] > limite_superior)
].copy()

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)
print("Cantidad de atípicos:", len(atipicos))

Q1: 311472.5
Q3: 4972182.458463397
IQR: 4660709.958463397
Límite inferior: -6679592.437695095
Límite superior: 11963247.396158492
Cantidad de atípicos: 46


In [ ]:
columnas_revision = [
    "codigoOC",
    "Institucion",
    "MecanismoCompra",
    "MonedaOC",
    "MontoNetoOC_CLP",
    "EstadoOC"
]

display(
    atipicos[columnas_revision]
    .sort_values("MontoNetoOC_CLP", ascending=False)
    .head(15)
)

,codigoOC,Institucion,MecanismoCompra,MonedaOC,MontoNetoOC_CLP,EstadoOC
743,1003780-2-SE26,GOBIERNO REGIONAL REGION METROPOLITANA,TratoDirecto,CLP,2.016712e+09,ACEPTADA
703,1261-2-SE26,GOBIERNO REGIONAL REGION METROPOLITANA,Licitacion,CLP,3.244430e+08,ACEPTADA
248,624-379-CM25,GOBIERNO REGIONAL IV REGION COQUIMBO,ConvenioMarco,CLP,2.880756e+08,ACEPTADA
919,836-246-SE25,GOBIERNO REGIONAL V REGION,Licitacion,CLP,2.857680e+08,RECEPCION CONFORME
391,624-67-SE26,GOBIERNO REGIONAL IV REGION COQUIMBO,Licitacion,CLP,2.826382e+08,RECEPCION CONFORME
927,549122-8-SE26,GOBIERNO REGIONAL V REGION,Licitacion,CLF,2.568121e+08,ACEPTADA
725,1260-36-SE26,GOBIERNO REGIONAL REGION METROPOLITANA,Licitacion,CLP,1.076250e+08,RECEPCION CONFORME
373,624-27-SE26,GOBIERNO REGIONAL IV REGION COQUIMBO,Licitacion,CLP,1.013160e+08,ACEPTADA
715,1261-6-SE26,GOBIERNO REGIONAL REGION METROPOLITANA,Licitacion,CLP,1.001000e+08,ACEPTADA
716,1261-7-SE26,GOBIERNO REGIONAL REGION METROPOLITANA,Licitacion,CLP,9.670240e+07,ACEPTADA


In [ ]:
codigos = (
    atipicos
    .sort_values("MontoNetoOC_CLP", ascending=False)
    ["codigoOC"]
    .head(5)
)

detalle = (
    df[df["codigoOC"].isin(codigos)]
    .drop_duplicates(subset=["codigoOC"])
)

display(
    detalle[
        [
            "codigoOC",
            "NombreOC",
            "DescripcionOC",
            "MontoNetoOC_CLP",
            "MonedaOC",
            "EstadoOC"
        ]
    ]
)

,codigoOC,NombreOC,DescripcionOC,MontoNetoOC_CLP,MonedaOC,EstadoOC
248,624-379-CM25,Orden de Compra generada por la Gran Compra: 7...,ADQUISICION CAMIONETAS PATRULLAJES PREVENTIVOS...,2.880756e+08,CLP,Aceptada
391,624-67-SE26,ADQUISICIÓN DE CAMIONES ALJIBES PARA EL TRANSP...,ADQUISICIÓN DE CAMIONES ALJIBES LA SERENA DESD...,2.826382e+08,CLP,Recepcion Conforme
703,1261-2-SE26,Conservación Edificio Institucional desde 1261...,"Solicitud de Compra N°11533, autorizada el 03 ...",3.244430e+08,CLP,Aceptada
743,1003780-2-SE26,Mantención Sistema de Control Transito,"Mantención Sistema de Control Transito, UOCTMo...",2.016712e+09,CLP,Aceptada
919,836-246-SE25,ORDEN DE COMPRA DESDE 836-6-LR25. Servicio Gua...,Servicio Guardias de Seguridad GORE Valparaíso...,2.857680e+08,CLP,Recepcion Conforme


In [ ]:

# Distribución de atípicos por mecanismo
resumen_atipicos = pd.crosstab(
    ordenes["MecanismoCompra"],
    ordenes["codigoOC"].isin(atipicos["codigoOC"])
)

resumen_atipicos.columns = [
    "No atípicas", "Atípicas"
]

display(resumen_atipicos)


,No atípicas,Atípicas
MecanismoCompra,,
CompraAgil,151,0
ConvenioMarco,85,4
Licitacion,36,32
TratoDirecto,18,10


_Tu conclusión sobre los atípicos:_



Mediante el método del rango intercuartílico (IQR), se identificaron 46 órdenes de compra con montos netos atípicos, equivalentes al 13,69 % de las 336 órdenes analizadas. El límite superior calculado fue de aproximadamente $11.963.247 CLP.

Se examinaron las cinco órdenes con mayores montos, contrastando sus valores con las descripciones y mecanismos de contratación. Estas corresponden a proyectos de infraestructura, mantención de sistemas, adquisición de vehículos y servicios de seguridad, por lo que sus importes resultan plausibles según la naturaleza de las contrataciones.

Sin embargo, la plausibilidad no constituye una validación definitiva de exactitud. Por esta razón, se decidió conservar los 46 valores atípicos sin modificarlos y declarar como limitación la falta de verificación documental de la totalidad de los casos.

---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

In [ ]:
# Tu código aquí

ordenes["mes_compra"] = ordenes["FechaEnvioOC"].dt.month

display(
    ordenes[
        ["codigoOC", "FechaEnvioOC", "mes_compra"]
    ].head(10)
)

print(
    ordenes["mes_compra"].value_counts().sort_index()
)


,codigoOC,FechaEnvioOC,mes_compra
0,624-5-AG26,2026-02-11,2
1,624-9-AG26,2026-02-13,2
4,624-11-AG26,2026-02-23,2
5,624-26-AG26,2026-03-13,3
6,624-37-AG26,2026-03-27,3
7,624-45-AG26,2026-04-02,4
8,624-52-AG26,2026-04-21,4
9,624-56-AG26,2026-04-22,4
10,624-57-AG26,2026-04-22,4
11,624-58-AG26,2026-04-22,4


mes_compra
1    16
2    46
3    58
4    70
5    65
6    81
Name: count, dtype: int64


_Por qué esta variable aporta a mi pregunta:_

Se construyó el atributo derivado `mes_compra` a partir de la variable `FechaEnvioOC`, extrayendo el número del mes de cada orden de compra.

Esta variable permite agrupar los registros mensualmente y analizar cómo se distribuyen los montos netos de las órdenes de compra durante el primer semestre de 2026. Además, facilita la comparación temporal entre los tres Gobiernos Regionales y sus mecanismos de contratación, contribuyendo directamente a responder la pregunta analítica del proyecto.



### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [ ]:
# Tu código aquí. Referencia:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# df_cod = pd.get_dummies(df, columns=['TU_COLUMNA_CATEGORICA'], drop_first=True)


# Codificación de la variable categórica MecanismoCompra

ordenes_cod = pd.get_dummies(
    ordenes,
    columns=["MecanismoCompra"],
    drop_first=True,
    dtype=int
)

print([
    columna for columna in ordenes_cod.columns
    if columna.startswith("MecanismoCompra_")
])

print("Antes:", ordenes.shape)
print("Después:", ordenes_cod.shape)


['MecanismoCompra_ConvenioMarco', 'MecanismoCompra_Licitacion', 'MecanismoCompra_TratoDirecto']
Antes: (336, 82)
Después: (336, 84)


---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [ ]:

def limpiar(rutas, sep=";", semilla=20262):
    """Prepara órdenes de ChileCompra.
    Devuelve (df_limpio, informe).
    """
    np.random.seed(semilla)

    tablas = []
    mecanismos = [
        "CompraAgil", "ConvenioMarco",
        "Licitacion", "Licitaciones", "TratoDirecto"
    ]

    # 1. Adquisición e integración
    for archivo in sorted(rutas):
        ruta = Path(archivo)

        d = pd.read_csv(
            ruta, sep=sep, encoding="cp1252"
        )

        d["RegionOrigen"] = ruta.parent.name

        mecanismo = next(
            (m for m in mecanismos
             if ruta.stem.endswith(m)), None
        )

        if mecanismo is None:
            raise ValueError(f"Mecanismo desconocido: {ruta.name}")

        d["MecanismoCompra"] = mecanismo.replace(
            "Licitaciones", "Licitacion"
        )

        tablas.append(d)

    original = pd.concat(tablas, ignore_index=True)

    informe = {
        "archivos": len(rutas),
        "filas_iniciales": len(original),
        "duplicados_exactos": int(
            original.duplicated().sum()
        ),
        "ordenes_unicas": int(
            original["codigoOC"].nunique()
        )
    }

    # 2. Centinelas: conservar los valores.
    # No se confirmaron centinelas en las
    # variables monetarias examinadas.

    # 3. Normalizar categorías
    original["EstadoOC"] = (
        original["EstadoOC"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.normalize("NFKD")
        .str.encode("ascii", "ignore")
        .str.decode("ascii")
    )

    # 4. Validar atributos por orden
    columnas_orden = [
        "FechaEnvioOC", "Institucion",
        "RegionOrigen", "MecanismoCompra",
        "MonedaOC", "MontoTotalOC",
        "MontoNetoOC_CLP", "EstadoOC"
    ]

    conteos = (
        original.groupby("codigoOC")[columnas_orden]
        .nunique(dropna=False)
    )

    if (conteos > 1).any().any():
        raise ValueError(
            "Hay atributos contradictorios dentro de una orden"
        )

    # 5. Consolidar a nivel de orden
    d = original[
        ["codigoOC"] + columnas_orden
    ].drop_duplicates(
        subset=["codigoOC"]
    ).copy()

    # 6. Tipos de dato
    d["FechaEnvioOC"] = pd.to_datetime(
        d["FechaEnvioOC"],
        format="%Y-%m-%d",
        errors="raise"
    )

    # Los montos ya fueron interpretados
    # como float64 durante la adquisición.

    # 7. Detectar atípicos sin eliminarlos
    col = "MontoNetoOC_CLP"

    q1 = d[col].quantile(0.25)
    q3 = d[col].quantile(0.75)
    iqr = q3 - q1

    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr

    informe["atipicos_conservados"] = int(
        ((d[col] < inferior) |
         (d[col] > superior)).sum()
    )

    # 8. Atributo derivado
    d["mes_compra"] = d["FechaEnvioOC"].dt.month

    # 9. Codificación categórica
    indicadores = pd.get_dummies(
        d["MecanismoCompra"],
        prefix="MecanismoCompra",
        drop_first=True,
        dtype=int
    )

    d = pd.concat([d, indicadores], axis=1)

    informe["filas_finales"] = len(d)
    informe["columnas_finales"] = len(d.columns)
    informe["ordenes_duplicadas_finales"] = int(
        d["codigoOC"].duplicated().sum()
    )

    return d, informe


In [ ]:
# Comprobar reproducibilidad
a, informe_a = limpiar(archivos_csv)
b, informe_b = limpiar(archivos_csv)

print("¿Reproducible?", a.equals(b))
print("¿Informes idénticos?", informe_a == informe_b)

¿Reproducible? True
¿Informes idénticos? True


In [ ]:
# Recuperar el conjunto limpio
limpio = a

# Comprobar
print("Dimensiones:", limpio.shape)
display(limpio.head())

Dimensiones: (336, 13)


,codigoOC,FechaEnvioOC,Institucion,RegionOrigen,MecanismoCompra,MonedaOC,MontoTotalOC,MontoNetoOC_CLP,EstadoOC,mes_compra,MecanismoCompra_ConvenioMarco,MecanismoCompra_Licitacion,MecanismoCompra_TratoDirecto
0,624-5-AG26,2026-02-11,GOBIERNO REGIONAL IV REGION COQUIMBO,Coquimbo,CompraAgil,CLP,1599700.34,1344286.0,RECEPCION CONFORME,2,0,0,0
1,624-9-AG26,2026-02-13,GOBIERNO REGIONAL IV REGION COQUIMBO,Coquimbo,CompraAgil,CLP,4230450.00,3555000.0,RECEPCION CONFORME,2,0,0,0
4,624-11-AG26,2026-02-23,GOBIERNO REGIONAL IV REGION COQUIMBO,Coquimbo,CompraAgil,CLP,475997.62,399998.0,RECEPCION CONFORME,2,0,0,0
5,624-26-AG26,2026-03-13,GOBIERNO REGIONAL IV REGION COQUIMBO,Coquimbo,CompraAgil,CLP,535500.00,450000.0,RECEPCION CONFORME,3,0,0,0
6,624-37-AG26,2026-03-27,GOBIERNO REGIONAL IV REGION COQUIMBO,Coquimbo,CompraAgil,CLP,1773100.00,1490000.0,ACEPTADA,3,0,0,0


In [ ]:
# Guardar el conjunto preparado
ruta_salida = "/content/mi_conjunto_limpio.csv"

limpio.to_csv(
    ruta_salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado:", ruta_salida)
print("Dimensiones:", limpio.shape)

Archivo guardado: /content/mi_conjunto_limpio.csv
Dimensiones: (336, 13)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:** https://github.com/Sayrinix/chilecompra-calidad-datos/tree/main

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_